In [28]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display, Markdown
# package loading

In [34]:
# 데이터 전처리

# 데이터 로드
file_path = r'C:\Users\esang\Documents\work\devel\kdy\data\raw\gss2024.csv'
df = pd.read_csv(file_path)

### gss2024 '정치 뉴스 소비 빈도' 특성에 따라 H1(독립표본 t-검정) 과 H3(일원분산분석) 선정

1. H1: 독립표본 t-검정 (Independent-samples t test)
    - 가설 : 성별(`sex`)에 따라 정치 뉴스 소비 빈도(`polnews`)에 통계적으로 유의미한 가치가 있을 것이라 판단.
    - t-검정은 2개의 집단으로 나뉘는 독립변수가 필요하다. sex변수는 1(남성)과 2(여성)으로 완벽하게 이분화되어 있어 t-검정 적용 대상이다.
    - "남성과 여성 중 누가 정치 뉴스를 더 자주 볼까?" 라는 직관적인 해석이 가능하다.
2. H3: 일원분산분석 (One-way ANOVA)
    - 가설 : 최종 학력(`degree`)수준에 따라 정치 뉴스 소비 빈도에 유의미한 차이가 있을 것이라 판단.
    - ANOVA는 3개 이상 집단의 평균을 비교할 때 사용한다. `degree` 변수는 O(고졸 미만)부터 4(대학원)까지 5개의 뚜렷한 범주를 가지고 있어 학력이라는 계층 구간에 따라 뉴스 소비량이 어떻게 변하는지 뜯어보기에 가장 알맞다.

In [24]:
# 결측치 (-1, 998, 999) NaN 처리
df.replace([-1, 998, 999], np.nan, inplace=True)

# 종속 변수 전처리: 숫자가 클수록 '자주 봄'이 되도록 역코딩 (원래는 1이 가장 자주 봄)
# 1~7 척도이므로 8에서 빼주면 완벽하게 뒤집힘
df['polnews_rev'] = 8 - df['polnews']

전처리 확인.



## H1. 독립표본 t-검정 (성별 -> 뉴스 소비 빈도)

In [30]:
display(Markdown("### [H1] 성별에 따른 정치 뉴스 소비 빈도 차이 (t-test)"))

# 분석에 필요한 변수만 결측치 제거
h1_df = df.dropna(subset=['sex', 'polnews_rev']).copy()
group_male = h1_df[h1_df['sex'] == 1]['polnews_rev']
group_female = h1_df[h1_df['sex'] == 2]['polnews_rev']

# 1. Levene 등분산 검정

# 비교하려는 두 집단의 데이터가 퍼져있는 정도(분산)가 비슷한지 확인하는 사전 검사
# t-검정이나 분산분석(anova)은 기본적으로 "집단들의 데이터 퍼짐 정도가 비슷하다"는 전제 조건이 필요하다. levene 검정을 통해 이 조건이 맞는지 먼저 확인 후 분산이 비슷하다면 일반적인 방식을, 분산이 다르다면 수식이 보정된 방식(welch's test)을 선택하여 분석의 정확도를 높이기 위해 사용한다. (pdf에 있음)

levene_stat, levene_p = stats.levene(group_male, group_female)
equal_var = levene_p >= 0.05

# 2. t-검정 (등분산 가정 충족 여부에 따라 자동으로 방식 결정)
t_stat, p_val = stats.ttest_ind(group_male, group_female, equal_var=equal_var)

# 3. Cohen's d (효과 크기) 계산

# 두 집단의 평균 차이가 실질적으로 얼마나 '큰' 차이인지를 보여주는 지표
# t-검정 결과(p-value)는 두 집단 간에 차이가 우연인지 아닌지만 알려준다. cohen's d를 통해 차이가 현실적으로 의미가 있는지를 객관적인 수치로 확인하기 위해 사용한다. (pdf에 있음)

n1, n2 = len(group_male), len(group_female)
var1, var2 = group_male.var(), group_female.var()
pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
cohens_d = (group_male.mean() - group_female.mean()) / pooled_std

t_test_results = pd.DataFrame({
    '집단': ['남성', '여성'],
    '표본 수(N)': [n1, n2],
    '평균 점수': [round(group_male.mean(), 2), round(group_female.mean(), 2)],
    '표준편차': [round(group_male.std(), 2), round(group_female.std(), 2)],
    't-통계량': [round(t_stat, 4), '-'],
    'p-value': [f"{p_val:.4f}", '-'],
    "Cohen's d": [round(cohens_d, 4), '-']
})

display(t_test_results)

### [H1] 성별에 따른 정치 뉴스 소비 빈도 차이 (t-test)

,집단,표본 수(N),평균 점수,표준편차,t-통계량,p-value,Cohen's d
0,남성,883,5.35,1.92,4.1018,0.0000,0.1897
1,여성,978,4.97,2.08,-,-,-


### H1: 성별에 따른 정치 뉴스 소비 빈도 차이 (독립표본 t-검정)

1. 결과 요약
    * 평균 차이: 남성의 뉴스 소비 빈도 점수(5.35점)가 여성(4.97점)보다 높게 나타났다.
    * 통계적 유의성 (p=0.0000): p값이 기준치(0.05)보다 작으므로 이 차이는 우연히 발생한 것이 아니라 통계적으로 유의미한 차이이다.
    * 효과 크기 (Cohen's d = 0.1897): 두 집단의 차이가 현실에서 얼마나 뚜렷한지 보여주는 지표이다. 보통 0.2 부근이면 '작은 효과'로 해석한다.

2. 해석
    * "분석 결과, 남성이 여성보다 정치 뉴스를 더 자주 소비하는 경향이 통계적으로 확인되었다."
    * "다만, 효과 크기(Cohen's d)가 0.1897로 작은 편이어서 두 성별 간의 뉴스 소비량 격차가 현실에서 극단적으로 크게 벌어지는 수준은 아님을 알 수 있다."
---

## H3. 일원분산분석 (최종 학력 -> 뉴스 소비 빈도)

In [33]:
display(Markdown("### [H3] 최종 학력에 따른 정치 뉴스 소비 빈도 차이 (One-way ANOVA)"))

h3_df = df.dropna(subset=['degree', 'polnews_rev']).copy()

# 보기 편하게 학력 레이블 매핑
degree_labels = {0: '고졸 미만', 1: '고졸', 2: '전문대졸', 3: '학사', 4: '대학원'}
h3_df['degree_label'] = h3_df['degree'].map(degree_labels)

# 학력별로 데이터 그룹화
groups = [group['polnews_rev'].values for name, group in h3_df.groupby('degree')]

# 1. Levene 등분산 검정
levene_stat_a, levene_p_a = stats.levene(*groups)

# 2. ANOVA (F-검정)
f_stat, p_val_a = stats.f_oneway(*groups)

# 3. Eta-squared (η², 효과 크기) 계산
overall_mean = h3_df['polnews_rev'].mean()
ss_between = sum(len(g) * (np.mean(g) - overall_mean)**2 for g in groups)
ss_total = sum((h3_df['polnews_rev'] - overall_mean)**2)
eta_sq = ss_between / ss_total

anova_results = pd.DataFrame({
    '검정 방법': ['One-way ANOVA'],
    '총 표본 수(N)': [len(h3_df)],
    'F-통계량': [round(f_stat, 4)],
    'p-value': [f"{p_val_a:.4f}"],
    'Eta-squared(η²)': [round(eta_sq, 4)]
})

display(anova_results)
display(Markdown("---"))

# 4. 사후 분석 (Tukey's HSD) - ANOVA 결과가 유의미할 때만(p < 0.05) 실행
if p_val_a < 0.05:
    print("▼ 사후검정(Tukey HSD): 어느 학력 집단 간에 차이가 났는가?")
    tukey = pairwise_tukeyhsd(endog=h3_df['polnews_rev'], groups=h3_df['degree_label'], alpha=0.05)
    tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    display(tukey_df)


### [H3] 최종 학력에 따른 정치 뉴스 소비 빈도 차이 (One-way ANOVA)

,검정 방법,총 표본 수(N),F-통계량,p-value,Eta-squared(η²)
0,One-way ANOVA,1865,10.8787,0.0000,0.0229


---

▼ 사후검정(Tukey HSD): 어느 학력 집단 간에 차이가 났는가?


,group1,group2,meandiff,p-adj,lower,upper,reject
0,고졸,고졸 미만,-0.5606,0.0104,-1.0321,-0.0891,True
1,고졸,대학원,0.4843,0.0048,0.1034,0.8653,True
2,고졸,전문대졸,-0.1409,0.9151,-0.5939,0.3122,False
3,고졸,학사,0.4276,0.0031,0.1027,0.7525,True
4,고졸 미만,대학원,1.0449,0.0000,0.5005,1.5893,True
5,고졸 미만,전문대졸,0.4197,0.3072,-0.1774,1.0168,False
6,고졸 미만,학사,0.9882,0.0000,0.4814,1.4950,True
7,대학원,전문대졸,-0.6252,0.0110,-1.1537,-0.0967,True
8,대학원,학사,-0.0567,0.9962,-0.4805,0.3671,False
9,전문대졸,학사,0.5685,0.0134,0.0788,1.0581,True


### H3: 최종 학력에 따른 정치 뉴스 소비 빈도 차이 (일원분산분석)

1. 결과 요약
    * 통계적 유의성 (p=0.0000): 5개의 학력 집단 중 적어도 어느 두 집단 사이에는 뉴스 소비 빈도에 분명한 차이가 존재한다.
    * 효과 크기 (Eta-squared = 0.0229): 응답자들의 정치 뉴스 소비량 차이 중 약 2.3% 정도가 '학력'이라는 요인에 의해 설명된다는 뜻이다. (작은 효과)

2. 사후검정(Tukey HSD) 상세 해석
    * 표에서 맨 오른쪽 reject = True로 나온 항목들이 '서로 확실하게 차이가 나는 집단들' 이다. 이를 바탕으로 뉴스 소비량을 계층화하면 다음과 같다.
        - 상위 그룹 (학사, 대학원): 이 두 집단 사이에는 차이가 없다(reject=False). 반면, 고졸이나 전문대졸 집단보다는 뉴스를 유의미하게 더 많이 본다.
        - 중위 그룹 (고졸, 전문대졸): 이 두 집단 사이에도 유의미한 차이가 없다. 상위 그룹보다는 뉴스를 덜 본다.
        - 하위 그룹 (고졸 미만): 고졸이나 4년제 대졸(학사/대학원) 집단에 비해 뉴스를 가장 적게 보는 것으로 나타났다.

3. 해석
    - "학력에 따라 정치 뉴스 소비 빈도에 통계적으로 유의미한 차이가 나타났다. 사후검정 결과, 학력이 높을수록 뉴스를 더 자주 소비하는 뚜렷한 계단식 경향성을 확인했다.
    - '학사 및 대학원' 집단이 뉴스를 가장 많이 소비했으며, '고졸 및 전문대졸' 집단이 그 뒤를 이었고, '고졸 미만' 집단의 소비 빈도가 가장 낮았다.
    - 결과적으로 고등 교육 수준이 정치 뉴스와의 접촉 빈도를 높이는 유의미한 요인임을 알 수 있다."


## 결과 종합
1. 분석 결과의 종합적 의미 (What the results mean together)
* 본 분석은 성별과 최종 학력이라는 핵심 인구통계학적 요인이 미국 성인들의 정치 뉴스 소비 빈도에 미치는 영향을 검증했다.
* 분석 결과, 남성이 여성보다 정치 뉴스를 더 자주 소비하며(H1), 학력 수준이 높아질수록 정치 뉴스 소비량이 뚜렷한 계단식 상승세를 보이는 것(H3)으로 나타났다. 두 요인 모두 통계적으로 매우 유의미한 차이를 보였으며, 이는 정치 뉴스에 대한 관심과 접근성이 사회경제적 배경 및 인구통계학적 특성에 따라 구조적으로 편향되어 있을 가능성을 시사한다. 특히 고등 교육(학사 및 대학원)이 정치 및 공공 문제에 대한 관여도를 높이는 중요한 매개체 역할을 하고 있음을 확인했다.

2. 분석의 한계점 (What the analysis cannot tell you)
* 첫째, 통계적 유의성(p<0.05)은 확보되었으나 두 분석 모두 효과 크기(Cohen's d=0.1897, Eta-squared=0.0229)가 작게 나타났다. 이는 성별이나 학력만으로는 개인의 뉴스 소비 빈도 차이를 완벽히 설명할 수 없으며, 개인의 정치적 성향, 소득 수준, 혹은 직업과 같은 다른 은닉 변수(Lurking variables)가 더 큰 영향을 미칠 수 있음을 의미한다.
* 둘째, 본 분석은 '정치 뉴스를 얼마나 자주 보는가(빈도)'만을 측정했을 뿐, '어떤 성향의 뉴스'를 소비하는지는 알 수 없다. 응답자가 편향된 파티잔(Partisan) 매체만 편식하는지, 혹은 다양한 매체를 교차 검증하며 소비하는지에 대한 뉴스의 질적 소비 행태는 파악할 수 없다.
* 셋째, 두 변수 간의 연관성만 확인했을 뿐 명확한 인과관계를 증명하지는 못한다. 예를 들어, 학력이 높아서 뉴스를 많이 보는 것인지, 아니면 뉴스를 많이 보는 지적 호기심을 가진 사람들이 고학력을 취득하게 된 것인지 그 선후 관계를 단언하기 어렵다.